# DOOM Sample Factory Pipeline (v4)

Changes from v3:
- Fixed Section 7 (the encoder-inspection cell). `make_doom_env_impl` and `create_env`
  both return a plain `Box` observation space, but `VizdoomEncoder` expects a `Dict`
  with an `"obs"` key -- that dict-wrapping happens deeper inside Sample Factory's model
  construction, not at the environment level. We now build the expected dict shape
  manually before calling `make_vizdoom_encoder`.

**Run this on Google Colab or Kaggle (Linux-backed) -- not native Windows.**
Enable GPU first: Colab -> Runtime > Change runtime type > GPU.

### 1. Install system dependencies

In [1]:
!apt-get update -qq
!apt-get install -y build-essential zlib1g-dev libsdl2-dev libjpeg-dev nasm tar \
    libbz2-dev libgtk2.0-dev cmake git libfluidsynth-dev libgme-dev libopenal-dev \
    timidity libwildmidi-dev unzip ffmpeg libboost-all-dev liblua5.1-dev xvfb

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'liblua5.1-0-dev' instead of 'liblua5.1-dev'
build-essential is already the newest version (12.10ubuntu1).
zlib1g-dev is already the newest version (1:1.3.dfsg-3.1ubuntu2.2).
libsdl2-dev is already the newest version (2.30.0+dfsg-1ubuntu3.1).
libjpeg-dev is already the newest version (8c-2ubuntu11).
nasm is already the newest version (2.16.01-1build1).
tar is already the newest version (1.35+dfsg-3ubuntu0.4).
libbz2-dev is already the newest version (1.0.8-5.1ubuntu0.1).
libgtk2.0-dev is already the newest version (2.24.33-4ubuntu1.1).
cmake is already the newest version (3.28.3-1build7).
git is already the newest version (1:2.43.0-1ubuntu7.3).
libfluidsynth-dev is already the newest versi

### 2. Install Sample Factory and ViZDoom
This can take several minutes.

In [2]:
!pip install sample-factory
!pip install vizdoom

### 3. Force a CUDA-enabled PyTorch build, then GPU check
Installing other packages above can silently swap Colab's preinstalled CUDA PyTorch for a
CPU-only build. Force the CUDA build explicitly, then verify.

In [3]:
!pip install torch --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124


**Restart the runtime now** (Runtime > Restart session), then continue from the next cell
below. A fresh pip-installed torch build sometimes needs a clean kernel to take effect.

In [4]:
import torch

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
else:
    print("No GPU visible -- fix the runtime/accelerator setting before continuing.")

torch version: 2.11.0+cu130
CUDA available: True
Device name: Tesla T4


### 4. The PyTorch 2.6 checkpoint-loading fix
Forces `torch.load` back to its pre-2.6 permissive default. Safe here because we only
ever load checkpoints trained by us, in this same notebook.

In [5]:
import torch

if not getattr(torch.load, "_is_patched", False):
    _original_torch_load = torch.load

    def _patched_torch_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return _original_torch_load(*args, **kwargs)

    _patched_torch_load._is_patched = True
    torch.load = _patched_torch_load

print("torch.load patched (weights_only=False).")

torch.load patched (weights_only=False).


### 5. Register Sample Factory's ViZDoom environments and models

In [6]:
import functools

from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.envs.create_env import create_env
from sample_factory.train import run_rl

from sf_examples.vizdoom.doom.doom_model import make_vizdoom_encoder
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec


def register_vizdoom_envs():
    for env_spec in DOOM_ENVS:
        make_env_func = functools.partial(make_doom_env_from_spec, env_spec)
        register_env(env_spec.name, make_env_func)


def register_vizdoom_models():
    """Registers Sample Factory's default ViZDoom-tuned CNN encoder.

    NOTE: default uses ELU and a GRU -- both overridden via config flags below
    (--nonlinearity=relu, --use_rnn=False) rather than needing a fully custom encoder,
    as long as BatchNorm is not also present (checked in Section 7).
    """
    global_model_factory().register_encoder_factory(make_vizdoom_encoder)


def register_vizdoom_components():
    register_vizdoom_envs()
    register_vizdoom_models()


def parse_vizdoom_cfg(argv=None, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser)
    doom_override_defaults(parser)
    final_cfg = parse_full_cfg(parser, argv)
    return final_cfg


register_vizdoom_components()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[2026-10-06 06:09:08,959][19983] register_encoder_factory: <function make_vizdoom_encoder at 0x7f4b12460e00>


### 6. Choose the scenario and training config

`--nonlinearity=relu`: overrides the ELU default -- required for ANN-to-SNN conversion
compatibility.

`--use_rnn=False` + `--env_framestack=4`: use frame-stacking for temporal info instead of
a GRU -- keeps the architecture a plain feedforward CNN, avoiding the much harder
recurrent-SNN conversion problem.

In [7]:
env = "doom_basic"
train_for_env_steps = 2000000   # raised from 500000 to resume training
num_workers = 8
num_envs_per_worker = 4

conversion_friendly_args = [
    f"--env={env}",
    "--experiment=conversion_friendly_v1",
    f"--num_workers={num_workers}",
    f"--num_envs_per_worker={num_envs_per_worker}",
    f"--train_for_env_steps={train_for_env_steps}",
    "--nonlinearity=relu",
    "--use_rnn=False",
]

### 7. Inspect the encoder BEFORE training

`create_env` returns a plain `Box` observation space, but `VizdoomEncoder` (the class
wrapping the actual CNN) expects a `Dict` with an `"obs"` key -- that wrapping normally
happens deeper inside Sample Factory's model-construction code, not at the environment
level. We build the expected dict shape manually here so we can inspect the encoder in
isolation.

Check the printed output for any `BatchNorm` layer -- if one appears, it needs to be
folded into the preceding conv layer after training (or replaced with a custom encoder)
before this model can be converted to an SNN.

In [8]:
inspect_cfg = parse_vizdoom_cfg(argv=conversion_friendly_args)
inspect_env = create_env(env, cfg=inspect_cfg, env_config=None)

# VizdoomEncoder expects obs_space["obs"] (and optionally obs_space["measurements"]),
# but create_env returns a plain Box, not a Dict -- build the dict shape it expects.
obs_space_for_encoder = {"obs": inspect_env.observation_space}

inspect_encoder = make_vizdoom_encoder(inspect_cfg, obs_space_for_encoder)

print(inspect_encoder)

has_batchnorm = any("BatchNorm" in type(m).__name__ for m in inspect_encoder.modules())
print("\nContains BatchNorm:", has_batchnorm)

inspect_env.close()

[2026-10-06 06:09:09,015][19983] Doom resolution: 160x120, resize resolution: (128, 72)
[2026-10-06 06:09:09,018][19983] ConvEncoder: input_channels=3
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[2026-10-06 06:09:09,391][19983] Conv encoder output size: 512
[2026-10-06 06:09:09,392][19983] Policy head output size: 512


VizdoomEncoder(
  (basic_encoder): ConvEncoder(
    (enc): RecursiveScriptModule(
      original_name=ConvEncoderImpl
      (conv_head): RecursiveScriptModule(
        original_name=Sequential
        (0): RecursiveScriptModule(original_name=Conv2d)
        (1): RecursiveScriptModule(original_name=ReLU)
        (2): RecursiveScriptModule(original_name=Conv2d)
        (3): RecursiveScriptModule(original_name=ReLU)
        (4): RecursiveScriptModule(original_name=Conv2d)
        (5): RecursiveScriptModule(original_name=ReLU)
      )
      (mlp_layers): RecursiveScriptModule(
        original_name=Sequential
        (0): RecursiveScriptModule(original_name=Linear)
        (1): RecursiveScriptModule(original_name=ReLU)
      )
    )
  )
)

Contains BatchNorm: False


### 8. Train
Only proceed once Section 7 confirms `Contains BatchNorm: False` -- otherwise, resolve
that first (see the closing notes).

In [9]:
cfg = parse_vizdoom_cfg(argv=conversion_friendly_args)
status = run_rl(cfg)

[2026-10-06 06:09:09,431][19983] Saved parameter configuration for experiment conversion_friendly_v1 not found!
[2026-10-06 06:09:09,432][19983] Starting experiment from scratch!
[2026-10-06 06:09:09,443][19983] Experiment dir /content/train_dir/conversion_friendly_v1 already exists!
[2026-10-06 06:09:09,444][19983] Resuming existing experiment from /content/train_dir/conversion_friendly_v1...
[2026-10-06 06:09:09,445][19983] Weights and Biases integration disabled
[2026-10-06 06:09:13,216][19983] Queried available GPUs: 0

[2026-10-06 06:09:13,219][19983] Environment var CUDA_VISIBLE_DEVICES is 0

[2026-10-06 06:09:17,044][19983] Automatically setting recurrence to 1
[2026-10-06 06:09:17,046][19983] Starting experiment with the following configuration:
help=False
algo=APPO
env=doom_basic
experiment=conversion_friendly_v1
train_dir=/content/train_dir
restart_behavior=resume
device=gpu
seed=None
num_policies=1
async_rl=True
serial_mode=False
batched_sampling=False
num_batches_to_accumul

### 9. Confirm training actually used the GPU and the correct overrides

In [10]:
import json

with open("train_dir/conversion_friendly_v1/config.json") as f:
    saved_cfg = json.load(f)

print("Device used for training:", saved_cfg.get("device"))
print("Nonlinearity used:", saved_cfg.get("nonlinearity"))
print("RNN used:", saved_cfg.get("use_rnn"))

Device used for training: gpu
Nonlinearity used: relu
RNN used: False


### 10. Evaluate (50 episodes)

In [16]:
import re, logging, numpy as np
from sample_factory.enjoy import enjoy

class Grab(logging.Handler):
    def __init__(self):
        super().__init__()
        self.avgs = []
    def emit(self, record):
        m = re.search(r"Avg episode reward: ([-\d.]+),", record.getMessage())
        if m:
            self.avgs.append(float(m.group(1)))

grab = Grab()
logging.getLogger("rl").addHandler(grab)   # assumes SF logs under "rl"

eval_cfg = parse_vizdoom_cfg(
    argv=[
        f"--env={env}",
        "--experiment=conversion_friendly_v1",
        "--num_workers=1",
        "--no_render",
        "--max_num_episodes=50",
        "--eval_deterministic=True",
        "--save_video",
    ],
    evaluation=True,
)
enjoy(eval_cfg)

a = np.array(grab.avgs)
n = np.arange(1, len(a) + 1)
eps = n * a - (n - 1) * np.concatenate([[0], a[:-1]])
print("episodes:", len(eps))
print(f"mean {eps.mean():.3f} ± {eps.std():.3f}")
print("episodes below 0:", (eps < 0).sum())
print("worst 5:", np.sort(eps)[:5].round(2))

[2026-10-06 06:24:15,504][19983] Loading existing experiment configuration from /content/train_dir/conversion_friendly_v1/config.json
[2026-10-06 06:24:15,505][19983] Overriding arg 'num_workers' with value 1 passed from command line
[2026-10-06 06:24:15,506][19983] Adding new argument 'no_render'=True that is not in the saved config file!
[2026-10-06 06:24:15,508][19983] Adding new argument 'save_video'=True that is not in the saved config file!
[2026-10-06 06:24:15,508][19983] Adding new argument 'video_frames'=1000000000.0 that is not in the saved config file!
[2026-10-06 06:24:15,509][19983] Adding new argument 'video_name'=None that is not in the saved config file!
[2026-10-06 06:24:15,510][19983] Adding new argument 'max_num_frames'=1000000000.0 that is not in the saved config file!
[2026-10-06 06:24:15,511][19983] Adding new argument 'max_num_episodes'=50 that is not in the saved config file!
[2026-10-06 06:24:15,512][19983] Adding new argument 'push_to_hub'=False that is not in

episodes: 50
mean 0.732 ± 0.091
episodes below 0: 0
worst 5: [0.58 0.61 0.62 0.64 0.64]


### 11. Watch the replay

In [17]:
from base64 import b64encode
from IPython.display import HTML

mp4 = open("train_dir/conversion_friendly_v1/replay.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(
    """
<video width=640 controls>
      <source src="%s" type="video/mp4">
</video>
"""
    % data_url
)

### 12. Record your results here

- Scenario: `doom_basic`
- Total training steps: 2,000,000
- nonlinearity: relu | use_rnn: False | single frame (no frame stacking).
- Contains BatchNorm: False
- Device used: GPU
- Evaluation episodes: 50
- Mean reward ± std: (fill in from `enjoy()`'s printed output)
- Checkpoint path: `train_dir/conversion_friendly_v1/checkpoint_p0/...`
- Date run: (fill in)

### Notes for the project going forward

- **Official Phase 1 ANN baseline pipeline**, replacing both the earlier SB3 notebook
  (unresolved evaluation discrepancy) and the v1/v2 Sample Factory notebooks (used
  default ELU + GRU, not conversion-friendly).
- **If Section 7 shows BatchNorm present**: either (a) fold BatchNorm into the preceding
  conv layer's weights after training (`torch.nn.utils.fuse_conv_bn_eval` can help), or
  (b) write and register a fully custom encoder matching `ConversionFriendlyCNN`'s
  constraints, via `global_model_factory().register_encoder_factory(...)`.
- **Reproducibility**: save the final checkpoint + `config.json` to Google Drive or
  download locally -- Colab/Kaggle sessions are ephemeral.

In [18]:
import shutil
from google.colab import files

shutil.make_archive("conversion_friendly_v1_backup", "zip", "train_dir/conversion_friendly_v1")
files.download("conversion_friendly_v1_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree(
    "train_dir/conversion_friendly_v1",
    "/content/drive/MyDrive/doom-snn-project/conversion_friendly_v1",
    dirs_exist_ok=True
)

Mounted at /content/drive


'/content/drive/MyDrive/doom-snn-project/conversion_friendly_v1'

In [5]:
!pip download sample-factory --no-deps --no-binary :all: -d /tmp/sf -q
!cd /tmp/sf && tar xzf sample_factory-*.tar.gz && grep -rniE "pool" sample_factory-*/sample_factory/model sample_factory-*/sf_examples/vizdoom --include=*.py

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
tar (child): sample_factory-*.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now
